In [13]:
from langchain_openai.chat_models import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.types import Send, RetryPolicy
from langgraph.func import entrypoint, task



from concurrent.futures import ThreadPoolExecutor, as_completed
from io import BytesIO
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import DocumentStream

import math
import operator
from typing import TypedDict, List, Optional, Annotated, Literal, Dict
from pydantic import BaseModel
import requests
from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
)
import os

from readability import Document
from markdownify import markdownify as md

import spacy
nlp = spacy.load("en_core_web_sm")

from dotenv import load_dotenv
load_dotenv('../.env')
EMAIL_ADDRESS = os.environ.get('EMAIL_ADDRESS')

In [2]:
# The PDF file containing the research paper
#from docling.document_converter import DocumentConverter

#converter = DocumentConverter()
#result = converter.convert("../data/simucell3d-nat-comp-sci-paper.pdf")

#markdown = result.document.export_to_markdown()

#with open("converted_pdf.md", "r") as file:
#    markdown = file.read()

#print(markdown)

In [2]:
import json
from pathlib import Path

file_dir = os.getcwd()
json_path = Path(os.path.join(file_dir, '../frontend/my-app/src/data/simucell3d-nat-comp-sci-paper-segmented-tokens.json'))

with json_path.open('r', encoding='utf-8') as f:
    segmented_text_tokens = json.load(f)

print(f'Loaded {len(segmented_text_tokens)} segmented tokens')
segmented_text_tokens[:3]

# Extract all sentences from the segmented text tokens and group them by sentence_id
sentences = {token["sentenceId"] : token["sentence"] for token in segmented_text_tokens}

full_text = "\n".join(sentences[sentence_id].replace("\n", " ") for sentence_id in sorted(sentences.keys()))
print(f'Full text length: {len(full_text)} characters')


Loaded 5152 segmented tokens
Full text length: 65613 characters


# References Extraction

Use the LLM to read the paper and extract the references one by one. For each reference, extract the title, authors, and DOI (if available). Then use these information to query the CrossRef API to get more metadata about the reference, such as the publication venue and year. 

In [3]:
# Create the model
llm_low_temp = ChatOpenAI(model="gpt-5-nano", temperature=0)

In [4]:
class Reference(BaseModel):
    ref_id: int
    journal: str
    title: str
    authors: Optional[List[str]] = None
    publication_year: Optional[int]
    doi: Optional[str] = ""

    # The URL that points to the paper landing page.
    html_url: Optional[str] = ""

    # The URL pointing directly to the pdf of the paper (better for data extraction)
    pdf_url: Optional[str] = ""

    is_open_access: Optional[bool] = None

    # The article content in markdown format
    content: Optional[str] = ""


class ReferenceExtractionState(TypedDict):

    # The document in markdown format
    document: str

    # Parsed and enriched references (replaced by downstream nodes).
    references: List[Reference]

In [5]:
import json
from langgraph.config import get_stream_writer


def reference_extraction_node(state: ReferenceExtractionState) -> ReferenceExtractionState:
    """
    Extract references from the bibliography section.
    """

    document = state.get("document", "")
    writer = get_stream_writer()

    system_prompt = """
You are an expert at analyzing references in scientific articles.
Find all references in the bibliography section.

Output format (strict):
- Output one JSON object per line (JSONL)
- No markdown
- No commentary
- No surrounding array

Each JSON object must follow:
{
  "ref_id": int,
  "journal": str,
  "title": str,
  "authors": [str] | null,
  "publication_year": int | null,
  "doi": str | null
}
"""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=document),
    ]

    partial_references: List[Reference] = []
    seen_keys = set()
    buffer = ""

    def emit_reference_line(line: str) -> None:
        """Extract the reference from the line and emit in real time"""
        line = line.strip().rstrip(",")
        if not line:
            return

        try:
            item = json.loads(line)
            ref = Reference.model_validate(item)
        except Exception:
            return

        key = (ref.ref_id, (ref.title or "").strip().lower())
        if key in seen_keys:
            return

        seen_keys.add(key)
        partial_references.append(ref)
        writer(
            {
                "event": "reference_extracted",
                "count": len(partial_references),
                "ref_id": ref.ref_id,
                "ref": ref.model_copy(),
            }
        )


    # Stream the raw output of the llm, parse each line, convert it 
    # in a reference object and emit it for further processing
    for chunk in llm_low_temp.stream(messages):
        content = chunk.content
        if not content:
            continue
        if isinstance(content, list):
            content = "".join(str(x) for x in content)

        buffer += content

        while "\n" in buffer:
            line, buffer = buffer.split("\n", 1)
            emit_reference_line(line)

    emit_reference_line(buffer)
    return {"references": partial_references}


builder = StateGraph(ReferenceExtractionState)
builder.add_node("reference_extraction_node", reference_extraction_node)
builder.add_edge(START, "reference_extraction_node")
builder.add_edge("reference_extraction_node", END)
graph = builder.compile()

initial_state: ReferenceExtractionState = {
    "document": full_text,
    "references": [],
}

final_state = None

# Stream the reference extraction in real time
for mode, data in graph.stream(initial_state, stream_mode=["custom", "updates"]):
    if mode == "custom" and data.get("event") == "reference_extracted":
        print(f"[{data['count']}] id={data['ref_id']} | {data['ref'].title}")
    elif mode == "updates":
        final_state = data

# Now final_state contains the complete output
references = final_state.get('reference_extraction_node', {}).get('references', [])
print(f"Total references extracted: {len(references)}")

[1] id=1 | An equatorial contractile mechanism drives cell elongation but not cell division.
[2] id=2 | Collective cell migration in morphogenesis, regeneration and cancer.
[3] id=3 | The extracellular matrix in development.
[4] id=4 | Forces in tissue morphogenesis and patterning.
[5] id=5 | Measuring mechanical stress in living tissues.
[6] id=6 | Measuring forces and stresses in situ in living tissues.
[7] id=7 | Microscale interrogation of 3D tissue mechanics.
[8] id=8 | The mechanical properties of the cell surface: III. The sea-urchin egg from fertilization to cleavage.
[9] id=9 | From molecules to cells: imaging soft samples with the atomic force microscope.
[10] id=10 | The optical stretcher: a novel laser tool to micromanipulate cells.
[11] id=11 | Mechanisms of pulsed laser ablation of biological tissues.
[12] id=12 | A mathematical model for outgrowth and spatial patterning of the vertebrate limb bud.
[13] id=13 | Video force microscopy reveals the mechanics of ventral furro

In [6]:
def fetch_crossref_metadata(ref: Reference) -> Reference:
    """
        Get the following metadata about the paper from crossref:
        - author names
        - journal name
        - html url
        - doi
    """
    crossref_request_params = {
                "query.title": ref.title,
                "rows": 5,
    }
    if ref.publication_year is not None:
        crossref_request_params["filter"] = (
            f"from-pub-date:{ref.publication_year}-01-01,"
            f"until-pub-date:{ref.publication_year}-12-31"
        )

    crossref_response = requests.get(
        "https://api.crossref.org/works",
        params=crossref_request_params,
        timeout=20,
    ).json()

    if crossref_response.get("message") and crossref_response["message"].get("items"):
        crossref_item = crossref_response["message"]["items"][0]

        # Get the URL to the paper
        ref.html_url = crossref_item.get("URL") or ref.html_url

        # Get its DOI to properly identify it
        ref.doi = crossref_item.get("DOI") or ref.doi

        # Get the journal name
        ref.journal = (
            (crossref_item.get("container-title") or [ref.journal])[0]
            if isinstance(crossref_item.get("container-title"), list)
            else crossref_item.get("container-title") or ref.journal
        )

        # Get the full author list, if available
        authors = []
        for author in crossref_item.get("author", []):
            given = author.get("given", "").strip()
            family = author.get("family", "").strip()
            full_name = f"{given} {family}".strip()
            if full_name:
                authors.append(full_name)
        if authors:
            ref.authors = authors
    return ref

def fetch_openaccess_metadata(ref: Reference) -> Reference:
    """
    Get the following metadata from unpaywall API:
    - is_openaccess
    - url_pdf (A URL directly pointing to the pdf download endpoint)
    """
    if not ref.doi:
        return ref

    unpaywall_url = f"https://api.unpaywall.org/v2/{ref.doi}?email={EMAIL_ADDRESS}"
    unpaywall_response = requests.get(unpaywall_url, timeout=20).json()

    ref.is_open_access = bool(unpaywall_response.get("is_oa", False))

    # Use OA location if present
    best_oa_location = unpaywall_response.get("best_oa_location") or {}
    if ref.is_open_access:
        ref.pdf_url = best_oa_location.get("url_for_pdf") or ref.pdf_url
        ref.html_url = best_oa_location.get("url_for_landing_page") or ref.html_url

    return ref

def fetch_paper_content(ref: Reference) -> Reference:
    """Load the content of the paper and save it as markdown file"""

    # Only load the content of openaccess articles
    if ref.is_open_access:

        # If we have access to the pdf download endpoint
        if ref.pdf_url:
            pdf_request_response = requests.get(ref.pdf_url, timeout=30)
            if pdf_request_response.status_code == 200:
                pdf_stream = BytesIO(pdf_request_response.content)
                converter = DocumentConverter()
                result = converter.convert(DocumentStream(name="paper.pdf", stream=pdf_stream))
                content = result.document.export_to_markdown()
                if len(content) > 5000:
                    ref.content = content

        # Instead try to download the article content directly from the HTML page
        if (not ref.content) and ref.html_url:
            html_request_response = requests.get(ref.html_url, timeout=30)

            if html_request_response.status_code == 200:
                doc = Document(html_request_response.text)
                article_html = doc.summary()
                content = md(article_html)
                if len(content) > 5000:
                    ref.content = content
    return ref

def fetch_reference_metadata_online(ref: Reference) -> Reference:
    """
    Enrich parsed references and return replacement list.
    """

    try:
        ref = fetch_crossref_metadata(ref)
        ref = fetch_openaccess_metadata(ref)
        ref = fetch_paper_content(ref)
    except Exception as e:
        print("Problem with reference:", ref.ref_id, "\n", type(e).__name__, e)
    return ref

enriched_references = list(map(fetch_reference_metadata_online, references))

An unexpected error occurred while opening the document paper.pdf
Traceback (most recent call last):
  File "/Users/srunser/Documents/Projects/CitationChecker/citation_checker_env/lib/python3.13/site-packages/docling/datamodel/document.py", line 177, in __init__
    self._init_doc(backend, path_or_stream)
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srunser/Documents/Projects/CitationChecker/citation_checker_env/lib/python3.13/site-packages/docling/datamodel/document.py", line 221, in _init_doc
    self._backend = backend(self, path_or_stream=path_or_stream)
                    ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srunser/Documents/Projects/CitationChecker/citation_checker_env/lib/python3.13/site-packages/docling/backend/docling_parse_backend.py", line 215, in __init__
    self._pdoc = pdfium.PdfDocument(self.path_or_stream, password=password)
                 ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srunser/Document

Problem with reference: 68 
 ConversionError Input document paper.pdf is not valid.


An unexpected error occurred while opening the document paper.pdf
Traceback (most recent call last):
  File "/Users/srunser/Documents/Projects/CitationChecker/citation_checker_env/lib/python3.13/site-packages/docling/datamodel/document.py", line 177, in __init__
    self._init_doc(backend, path_or_stream)
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srunser/Documents/Projects/CitationChecker/citation_checker_env/lib/python3.13/site-packages/docling/datamodel/document.py", line 221, in _init_doc
    self._backend = backend(self, path_or_stream=path_or_stream)
                    ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srunser/Documents/Projects/CitationChecker/citation_checker_env/lib/python3.13/site-packages/docling/backend/docling_parse_backend.py", line 215, in __init__
    self._pdoc = pdfium.PdfDocument(self.path_or_stream, password=password)
                 ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srunser/Document

Problem with reference: 85 
 ConversionError Input document paper.pdf is not valid.


An unexpected error occurred while opening the document paper.pdf
Traceback (most recent call last):
  File "/Users/srunser/Documents/Projects/CitationChecker/citation_checker_env/lib/python3.13/site-packages/docling/datamodel/document.py", line 177, in __init__
    self._init_doc(backend, path_or_stream)
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srunser/Documents/Projects/CitationChecker/citation_checker_env/lib/python3.13/site-packages/docling/datamodel/document.py", line 221, in _init_doc
    self._backend = backend(self, path_or_stream=path_or_stream)
                    ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srunser/Documents/Projects/CitationChecker/citation_checker_env/lib/python3.13/site-packages/docling/backend/docling_parse_backend.py", line 215, in __init__
    self._pdoc = pdfium.PdfDocument(self.path_or_stream, password=password)
                 ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srunser/Document

Problem with reference: 87 
 ConversionError Input document paper.pdf is not valid.


# Statements Extraction

After splitting the text of the paper into sentences, give each sentence in parallel to the LLM and ask it to classify whether the sentence is a statement that can be verified or not. If it is a statement, ask the LLM to extract the main claim of the statement and any relevant information that can be used for verification, such as a summary of the claim and the supporting citations.

In [7]:
class Statement(BaseModel):
    claim : Annotated[str, "One sentence summary of the scientific claim being made in the statement."]
    sentence_id : Annotated[int, "The id of the sentencefrom which the statement was extracted."]
    citations : Annotated[List[int], "The citation associated with the statement."]
    verification_status : Annotated[Literal["Supported", "Unsupported", "Unverified"], "The statement verification status."] = "Unverified"
    verification_explanation : Annotated[str, "A brief explanation of the reasoning behind the verification status."] = "Verification not yet performed."

# The global state used to track the extraction of statements from the text
class GlobalStatementExtractionState(TypedDict):

    # Contains the document 
    sentences : Dict[int, str]

    # The statement in the proper format
    statements: Annotated[List[Statement], operator.add]


# Substate used for the analysis of each sentence
class SentenceAnalysisState(TypedDict):

    # The id of the sentence being analyzed
    sentence_id : int

    # The text of the sentence being analyzed
    sentence : str

    # The other sentences in the document, to provide context for the analysis
    sentences : Dict[int, str]


In [9]:
from langgraph.config import get_stream_writer


MAX_CONCURRENCY = 10


def fan_out_statement_extraction_node(state: GlobalStatementExtractionState):
    """Parallelize the statement extraction by creating one node per sentence in the document"""
    return [
        Send(
            "statement_extraction_node",
            {
                "sentence_id": sentence_id,
                "sentence": sentence,
                "sentences": state["sentences"],
            },
        )
        for sentence_id, sentence in state["sentences"].items()
    ]


def statement_extraction_node(state: SentenceAnalysisState) -> GlobalStatementExtractionState:
    """Analyze a sentence and extract the scientific statement and its associated citations, if any"""

    sentence_id = state["sentence_id"]
    sentence = state["sentence"]
    sentences = state["sentences"]
    writer = get_stream_writer()

    # Get the text surrounding the chunk
    previous_sentence = sentences[sentence_id - 1] if sentence_id - 1 in sentences else ""
    next_sentence     = sentences[sentence_id + 1] if sentence_id + 1 in sentences else ""    
    surrounding_text = previous_sentence + sentence + next_sentence

    class ModelOutput(BaseModel):
        claim : Annotated[Optional[str], "One sentence summary of the scientific claim being made in the statement."] = None
        citations : Annotated[List[int], "The citation associated with the statement."]


    prompt = f"""
        Given the following sentence:
        {sentence}

        If this sentence a scientific statement supported by one or several citations,
        extract its associated citations and make a one line summary of the claim being made. 
        Do not use pronouns in your summary as a subject. These claim summary should be understandable on its own, 
        without the need to refer back to the original sentence.
        Here is the text surrounding the chunk to give you more context and help you understand the claim that is made:
        {surrounding_text}

        Rules:
        - if the chunk of text contains no citation, return nothing
        - if the chunk of text contains a scientific claim, write a one sentence summary of the claim
        - only include claims that have at least one citation number
        - preserve the exact citation numbers from the text
        - do not invent citations
        - do not include brackets or parentheses around citation numbers
        - no markdown
        - no numbering
    """

    result = llm_low_temp.with_structured_output(ModelOutput).invoke(prompt)
    if result.claim is None or len(result.citations) == 0:
        return {"statements": []}
    

    statement = Statement(
        claim=result.claim,
        sentence_id=sentence_id,
        citations=result.citations,
    )

    writer(
        {
            "event": "statement_extracted",
            "sentence_id": sentence_id,
            "statement": statement.model_dump(),
        }
    )

    return {"statements": [statement]}


statement_extraction_graph_builder = StateGraph(GlobalStatementExtractionState)
statement_extraction_graph_builder.add_node("statement_extraction_node", statement_extraction_node)
statement_extraction_graph_builder.add_conditional_edges(START, fan_out_statement_extraction_node)
statement_extraction_graph_builder.add_edge("statement_extraction_node", END)
statement_extraction_graph = statement_extraction_graph_builder.compile()


statement_extraction_input = {
    "sentences": sentences,
}

statement_extraction_res = None
runtime_config = {"max_concurrency": MAX_CONCURRENCY}

for mode, data in statement_extraction_graph.stream(
    statement_extraction_input,
    config=runtime_config,
    stream_mode=["custom", "values"],
):
    if mode == "custom" and data.get("event") == "statement_extracted":
        statement_data = data.get("statement", {})
        print(f"[sentence {data.get('sentence_id')}] {statement_data.get('claim')}")
    elif mode == "values":
        statement_extraction_res = data

print(f"Total statements extracted: {len(statement_extraction_res.get('statements', []))}")

[sentence 10] Cellular behaviors are regulated by the mechanical properties of both cells and extracellular matrix (ECM), along with the distribution of stresses within tissues 3 and 4.
[sentence 15] Advances in fluorescent microscopy, image processing and computation power enable complementing direct measurements with in silico models, leading to a more global understanding of cellular dynamics underlying tissue morphogenesis and homeostasis.
[sentence 9] Tissue shape is determined by the dynamic positioning of constituent cells, which can collectively deform or migrate to drive macroscopic changes in tissue morphologies 1,2.
[sentence 8] Disruptions in tissue morphology are associated with a range of pathological conditions including cancer and birth defects
[sentence 12] Various experimental methods including micropipette aspiration, atomic force microscopy, optical stretcher, and laser ablation have been developed to contribute to this understanding 5–7, 8, 9, 10 and 11.
[sentence 

# Statements Verification

Once the statements and references are extracted, we can verify each statement in parallel. For each statement, extract the content of the references that are cited in the statement and give them to the LLM along with the statement. Ask the LLM to determine whether the statement is supported, contradicted, or not addressed by the cited references. If the statement is supported or contradicted, ask the LLM to provide a brief explanation of its reasoning.

In [17]:
import re
import time
from openai import RateLimitError

# Substate used for the analysis of each sentence
class StatementVerificationState(TypedDict):

    # The id of the sentence being analyzed
    unverified_statements : List[Statement]
    verified_statements : Annotated[List[Statement], operator.add]

    # The references associated with statements
    references : Dict[int, Reference]

class StatementVerificationSubstate(TypedDict):

    # The statement being verified
    statement : Statement

    # The references associated with the statement
    references : Dict[int, Reference]

def fan_out_statement_verification_node(state: StatementVerificationState):
    """Parallelize the statement verification by creating one node per statement to verify"""
    return [
        Send(
            "statement_verification_node",
            {
                "statement": statement,
                "references": state["references"],
            },
        )
        for statement in state["unverified_statements"]
    ]


def extract_retry_after_seconds(err: Exception, default_seconds: int = 20) -> int:
    """Parse the provider hint: 'Please try again in 22.073s'."""
    message = str(err)
    match = re.search(r"Please try again in\s+([0-9.]+)s", message)
    if not match:
        return default_seconds
    try:
        # Add a 1-second safety buffer to avoid retrying too early.
        return max(1, min(int(float(match.group(1))) + 1, 120))
    except Exception:
        return default_seconds


def statement_verification_node(state: StatementVerificationSubstate) -> StatementVerificationState:
    """Verify a scientific statement by checking whether the cited references support or contradict it"""

    # Stream writer to send intermediate results to the frontend in real time
    writer = get_stream_writer()

    statement = state["statement"]
    references = state["references"]

    # There are different scenarios to consider:
    # - If none of the cited references is open access, then we cannot verify the statement.
    # - If at least one of the cited references is open access, we can try to verify the statement.
    reference_lst = [references[ref_id] for ref_id in statement.citations if ref_id in references]
    open_access_references = [ref for ref in reference_lst if ref.is_open_access]

    if len(open_access_references) == 0:
        # We cannot verify the statement, as none of the cited references is open access
        statement.verification_status = "Unverified"
        statement.verification_explanation = "None of the cited references are open access, so we cannot verify this statement."

        writer(
            {
                "event": "statement_verified",
                "sentence_id": statement.sentence_id,
                "verification_status": statement.verification_status,
                "verification_explanation": statement.verification_explanation,
            }
        )
        
        return {"verified_statements": [statement]}

    class ModelOutput(BaseModel):
        verification_status : Annotated[Literal["Supported", "Unsupported", "Unverified"], "The statement verification status."] = "Unverified"
        verification_explanation : Annotated[str, "A brief explanation of the reasoning behind the verification status."] = "Verification not yet performed."

    # Create the prompt to verify the statement
    prompt = f"""
You are an expert at verifying scientific claims by analyzing the cited references.
Here is the scientific claim you need to verify:
{statement.claim}

Here is the content of the cited references: {
    "\n\n".join(
        f"Reference {ref.ref_id}:\nTitle: {ref.title}\nContent: {ref.content if ref.content else 'Content not available'}"
        for ref in open_access_references
    )
}

Rules:
- If the content of the cited references clearly supports the claim, then the verification status is "Supported".
- If the content of the cited references clearly contradicts the claim, then the verification status is "Unsupported".
- If the content of the cited references does not provide clear evidence to support or contradict the claim, then the verification status is "Unverified".
- Provide a brief explanation of the reasoning behind the verification status, based on the content of the cited references.
"""

    try:
        result = llm_low_temp.with_structured_output(ModelOutput).invoke(prompt)
    except RateLimitError as e:
        # Wait for the provider-indicated cool-down, then re-raise so LangGraph RetryPolicy retries this node.
        wait_seconds = extract_retry_after_seconds(e, default_seconds=20)
        print(
            f"[sentence {statement.sentence_id}] rate-limited; sleeping {wait_seconds}s before retry"
        )
        time.sleep(wait_seconds)
        raise

    statement.verification_status = result.verification_status
    statement.verification_explanation = result.verification_explanation

    writer(
        {
            "event": "statement_verified",
            "sentence_id": statement.sentence_id,
            "verification_status": statement.verification_status,
            "verification_explanation": statement.verification_explanation,
        }
    )

    return {"verified_statements": [statement]}


# Create the statement verification graph
statement_verification_graph_builder = StateGraph(StatementVerificationState)
statement_verification_graph_builder.add_node(
    "statement_verification_node",
    statement_verification_node,
    retry_policy=RetryPolicy(retry_on=RateLimitError, max_attempts=5),
)
statement_verification_graph_builder.add_conditional_edges(START, fan_out_statement_verification_node)
statement_verification_graph_builder.add_edge("statement_verification_node", END)
statement_verification_graph = statement_verification_graph_builder.compile()

statement_verification_input = {
    "unverified_statements": statement_extraction_res["statements"],
    "verified_statements": [],
    "references": {ref.ref_id: ref for ref in enriched_references},
}

statement_verification_res = None
runtime_config = {"max_concurrency": 3}
for mode, data in statement_verification_graph.stream(
    statement_verification_input,
    config=runtime_config,
    stream_mode=["custom", "values"],
):
    if mode == "custom" and data.get("event") == "statement_verified":
        print(f"[sentence {data.get('sentence_id')}] verification status: {data.get('verification_status')} | explanation: {data.get('verification_explanation')}")
    elif mode == "values":
        statement_verification_res = data

print(f"Total statements verified: {len(statement_verification_res.get('verified_statements', []))}")

[sentence 8] verification status: Unverified | explanation: Reference 1 shows that disruptions in the actomyosin-based equatorial ring and actin turnover in Ciona notochord cells cause abnormal cell shapes and impaired elongation, i.e., morphological defects during tissue morphogenesis. However, it does not address cancer or human birth defects, nor does it provide evidence linking tissue morphology disruptions to these pathological conditions. Therefore, the reference does not clearly verify the claim as stated (it neither directly supports nor contradicts the association with cancer or birth defects in humans).
[sentence 9] verification status: Supported | explanation: Reference 1 shows that elongation of the notochord tissue in Ciona intestinalis is achieved through dynamic, cell-autonomous shape changes rather than cell division. Notochord cells remain postmitotic while undergoing an equatorial actomyosin ring–driven constriction, cortical actin/myosin flow toward the equator, and 